# create sp for logging

In [5]:
use [sample];
GO

SELECT DB_NAME() AS db_name;
GO

Commands completed successfully.

(1 row affected)

db_name
-------
sample 
(1 row)

## spLog - base sp for logging

In [94]:
CREATE OR ALTER PROCEDURE dbo.spLog
    @Level NVARCHAR(10),    -- failing to specify size defaults to 1 !!!
    @Message NVARCHAR(MAX)
AS
BEGIN
    SELECT
        GETDATE() as [Timestamp],
        @Level as [Level], 
        @Message as [Message];

    DECLARE @Msg NVARCHAR(MAX) = FORMATMESSAGE('%s | %s | %s', 
        CONVERT(NVARCHAR, GETDATE(), 120), -- Style 120 is the "Golden Standard"        UPPER(@Level), 
        @Message
    );
    PRINT @Msg;
END
GO


Commands completed successfully.

## add info, warning, error helpers

In [95]:
CREATE OR ALTER PROCEDURE dbo.spInfo
    @Message NVARCHAR(MAX)
AS
BEGIN
    EXEC dbo.spLog 'INFO', @Message;
END
GO


Commands completed successfully.

In [96]:
CREATE OR ALTER PROCEDURE dbo.spWarn
    @Message NVARCHAR(MAX)
AS
BEGIN
    EXEC dbo.spLog 'WARN', @Message;
END
GO


Commands completed successfully.

In [97]:
CREATE OR ALTER PROCEDURE dbo.spError
    @Message NVARCHAR(MAX)
AS
BEGIN
    DECLARE @ErrorMsg NVARCHAR(MAX) = ''
    -- SELECT ERROR_NUMBER(), ERROR_MESSAGE();
    IF ERROR_MESSAGE() IS NOT NULL
        SET @ErrorMsg = FORMATMESSAGE('%s: %i - %s', 
            @Message, 
            ERROR_NUMBER(), 
            ERROR_MESSAGE()
        );
    ELSE
        SET @ErrorMsg = @Message;
    

    EXEC dbo.spLog 'ERROR', @ErrorMsg;
END
GO


Commands completed successfully.

## test sps

In [98]:
SELECT DB_NAME() AS db_name;
GO

EXEC dbo.spLog @Level='INFO', @Message='hello'
EXEC dbo.spInfo @Message='hello'
EXEC dbo.spWarn @Message='hello'
EXEC dbo.spError @Message='hello'
GO


(1 row affected)

db_name
-------
sample 
(1 row)

(1 row affected)
2026-03-25 07:19:37 | hello | (null)
(1 row affected)
2026-03-25 07:19:37 | hello | (null)
(1 row affected)
2026-03-25 07:19:37 | hello | (null)
(1 row affected)
2026-03-25 07:19:37 | hello | (null)

Timestamp               | Level | Message
------------------------+-------+--------
2026-03-25 07:19:37.710 | INFO  | hello  
(1 row)

Timestamp               | Level | Message
------------------------+-------+--------
2026-03-25 07:19:37.713 | INFO  | hello  
(1 row)

Timestamp               | Level | Message
------------------------+-------+--------
2026-03-25 07:19:37.713 | WARN  | hello  
(1 row)

Timestamp               | Level | Message
------------------------+-------+--------
2026-03-25 07:19:37.713 | ERROR | hello  
(1 row)